# 22 · Target per cápita (consumo / población) — 2026-09-12

Analiza si la población municipal mejora el modelo de consumo urbano del simulador y de qué forma.
Resultado del análisis: como **feature de entrada empeora** (+1,0 pp de MAPE), pero como **target
per cápita** (predecir `consumo/población` y reconstruir × población) **mejora**.

Protocolo (idéntico a `include/ml/entrenar.py`): 67 GBMs por municipio, train < 2022, test 2022-2024
recursivo con lag actualizado. Al proyectar, la población se **congela en el año base** (réplica de
producción); la variante `pc_actual` usa la población real de cada año de test solo como cota
optimista de referencia.

Población: `data/censo_municipal_baleares.csv` (padrón municipal IBESTAT 1998-2025, cargado en
`gold.censo_municipal_baleares`; la tabla conserva el nombre "censo").

**Regla de decisión**: adoptar el target per cápita si el MAPE recursivo con población congelada
mejora ≥ 0,5 pp y la mejora es significativa municipal a municipal (Wilcoxon).

In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl
from scipy.stats import wilcoxon
from sklearn.ensemble import GradientBoostingRegressor

DATA = Path("data")
panel = pl.read_parquet(DATA / "panel_features.parquet")
pob = (
    pl.read_csv(DATA / "censo_municipal_baleares.csv", infer_schema_length=None)
    .with_columns(pl.col("cod_municipio_ine").cast(pl.Int64).alias("cod_municipio"))
    .select("cod_municipio", "anio", "poblacion")
)
panel = panel.join(pob, on=["cod_municipio", "anio"], how="left").sort(["cod_municipio", "anio"])
assert panel["poblacion"].null_count() == 0, "poblacion incompleta en el panel"

PARAMS = {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "random_state": 42}
TRAIN_DESDE, TEST_START = 2016, 2022
FEATURES = ["anio", "iph_media", "ocupacion_media", "lluvia_anual_mm"]
BASE_PC = FEATURES + ["lag1_pc"]

print("panel:", panel.shape, "| municipios:", panel["cod_municipio"].n_unique())
print("población:", pob["anio"].min(), "-", pob["anio"].max(), "| filas:", pob.height)

panel: (670, 11) | municipios: 67
población: 1998 - 2025 | filas: 1876


In [2]:
def evaluar_recursivo(g, modo, test_start=TEST_START, single_year=False):
    """modo: base | pc_frozen (produccion) | pc_actual (poblacion real en test, cota optimista)."""
    train = g.filter(
        (pl.col("anio") >= TRAIN_DESDE) & (pl.col("anio") < test_start) & pl.col("lag1").is_not_null()
    )
    test = g.filter(pl.col("anio") == test_start) if single_year else g.filter(pl.col("anio") >= test_start)
    test = test.sort("anio")
    if train.height < 4 or test.height == 0:
        return None
    if modo == "base":
        m = GradientBoostingRegressor(**PARAMS)
        m.fit(train.select(FEATURES + ["lag1"]).to_numpy(), train["consumo_hm3"].to_numpy())
        last = float(train["consumo_hm3"].to_list()[-1])
        pred = []
        for row in test.iter_rows(named=True):
            last = max(float(m.predict([[float(row[f]) for f in FEATURES] + [last]])[0]), 0.0)
            pred.append(last)
    else:
        tr = train.with_columns(
            (pl.col("consumo_hm3") / pl.col("poblacion")).alias("y_pc"),
            (pl.col("lag1") / pl.col("poblacion")).alias("lag1_pc"),
        )
        pob_base = float(train["poblacion"].to_list()[-1])
        m = GradientBoostingRegressor(**PARAMS)
        m.fit(tr.select(FEATURES + ["lag1_pc"]).to_numpy(), tr["y_pc"].to_numpy())
        last_pc = float(tr["y_pc"].to_list()[-1])
        pred = []
        for row in test.iter_rows(named=True):
            pob_t = pob_base if modo == "pc_frozen" else float(row["poblacion"])
            p = max(float(m.predict([[float(row[f]) for f in FEATURES] + [last_pc]])[0]), 0.0) * pob_t
            pred.append(p)
            last_pc = p / pob_t
    y = test["consumo_hm3"].to_numpy()
    return float(np.mean(np.abs((y - np.asarray(pred)) / y)) * 100)

In [3]:
partes = panel.partition_by("cod_municipio", as_dict=True)
filas = [
    {
        "cod_municipio": int(cod_t[0]),
        "base": evaluar_recursivo(g, "base"),
        "pc_frozen": evaluar_recursivo(g, "pc_frozen"),
        "pc_actual": evaluar_recursivo(g, "pc_actual"),
    }
    for cod_t, g in partes.items()
]
res = pl.DataFrame([f for f in filas if f["base"] is not None]).with_columns(
    delta_frozen=(pl.col("pc_frozen") - pl.col("base")),
    delta_actual=(pl.col("pc_actual") - pl.col("base")),
)
print(f"MAPE medio  base={res['base'].mean():.3f}  pc_frozen={res['pc_frozen'].mean():.3f}  pc_actual={res['pc_actual'].mean():.3f}")
print(f"pc_frozen: delta medio={res['delta_frozen'].mean():+.3f} pp | mediana={res['delta_frozen'].median():+.3f} | mejora {(res['delta_frozen'] < 0).sum()}/{res.height}")
print(f"pc_actual: delta medio={res['delta_actual'].mean():+.3f} pp | mediana={res['delta_actual'].median():+.3f} | mejora {(res['delta_actual'] < 0).sum()}/{res.height}")
w = wilcoxon(res["base"].to_numpy(), res["pc_frozen"].to_numpy())
print(f"Wilcoxon base vs pc_frozen: estadistico={w.statistic:.1f} p-valor={w.pvalue:.5f}")
res.sort("delta_frozen", descending=True).head(6).select("cod_municipio", "base", "pc_frozen", "delta_frozen")

MAPE medio  base=8.863  pc_frozen=7.948  pc_actual=7.587
pc_frozen: delta medio=-0.915 pp | mediana=-0.839 | mejora 43/67
pc_actual: delta medio=-1.276 pp | mediana=-1.409 | mejora 41/67
Wilcoxon base vs pc_frozen: estadistico=648.0 p-valor=0.00216


cod_municipio,base,pc_frozen,delta_frozen
i64,f64,f64,f64
7024,20.885406,26.084133,5.198728
7007,4.524178,8.994772,4.470594
7063,0.814224,4.645542,3.831318
7018,11.898399,15.619911,3.721512
7051,8.869852,12.578604,3.708751
7061,10.148299,12.745321,2.597022


In [4]:
def evaluar_pc(features, test_start=TEST_START):
    out = []
    for _, g in panel.partition_by("cod_municipio", as_dict=True).items():
        train = g.filter(
            (pl.col("anio") >= TRAIN_DESDE) & (pl.col("anio") < test_start) & pl.col("lag1").is_not_null()
        )
        test = g.filter(pl.col("anio") >= test_start).sort("anio")
        if train.height < 4 or test.height == 0:
            continue
        tr = train.with_columns(
            (pl.col("consumo_hm3") / pl.col("poblacion")).alias("y_pc"),
            (pl.col("lag1") / pl.col("poblacion")).alias("lag1_pc"),
        )
        pob_base = float(train["poblacion"].to_list()[-1])
        m = GradientBoostingRegressor(**PARAMS).fit(tr.select(features).to_numpy(), tr["y_pc"].to_numpy())
        last = float(tr["y_pc"].to_list()[-1])
        pred = []
        for row in test.iter_rows(named=True):
            x = [last if f == "lag1_pc" else float(row[f]) for f in features]
            p = max(float(m.predict([x])[0]), 0.0) * pob_base
            pred.append(p)
            last = p / pob_base
        y = test["consumo_hm3"].to_numpy()
        out.append(float(np.mean(np.abs((y - np.asarray(pred)) / y)) * 100))
    return float(np.mean(out))


print("pc completo        :", round(evaluar_pc(BASE_PC), 3))
print("pc sin ocupacion   :", round(evaluar_pc([f for f in BASE_PC if f != "ocupacion_media"]), 3))
print("pc sin IPH         :", round(evaluar_pc([f for f in BASE_PC if f != "iph_media"]), 3))
print("pc sin anio        :", round(evaluar_pc([f for f in BASE_PC if f != "anio"]), 3))
print("pc sin IPH ni ocup :", round(evaluar_pc([f for f in BASE_PC if f not in ("iph_media", "ocupacion_media")]), 3))

pc completo        : 7.948
pc sin ocupacion   : 9.569
pc sin IPH         : 10.307
pc sin anio        : 7.73
pc sin IPH ni ocup : 12.515


In [5]:
print("Walk-forward (1 año, pc_frozen vs base):")
for t in (2021, 2022, 2023, 2024):
    b = [evaluar_recursivo(g, "base", t, single_year=True) for _, g in panel.partition_by("cod_municipio", as_dict=True).items()]
    p = [evaluar_recursivo(g, "pc_frozen", t, single_year=True) for _, g in panel.partition_by("cod_municipio", as_dict=True).items()]
    b = [v for v in b if v is not None]
    p = [v for v in p if v is not None]
    print(f"  {t}: base={np.mean(b):.3f} per_capita={np.mean(p):.3f} delta={np.mean(p) - np.mean(b):+.3f} pp")

Walk-forward (1 año, pc_frozen vs base):
  2021: base=6.321 per_capita=6.115 delta=-0.207 pp
  2022: base=6.484 per_capita=5.429 delta=-1.055 pp
  2023: base=6.349 per_capita=6.107 delta=-0.242 pp
  2024: base=6.234 per_capita=6.393 delta=+0.159 pp


In [6]:
def elasticidad_pc(fe, feats=BASE_PC):
    acc = []
    for _, g in panel.partition_by("cod_municipio", as_dict=True).items():
        train = g.filter(
            (pl.col("anio") >= TRAIN_DESDE) & (pl.col("anio") <= 2024) & pl.col("lag1").is_not_null()
        )
        if train.height < 4:
            continue
        tr = train.with_columns(
            (pl.col("consumo_hm3") / pl.col("poblacion")).alias("y_pc"),
            (pl.col("lag1") / pl.col("poblacion")).alias("lag1_pc"),
        )
        m = GradientBoostingRegressor(**PARAMS).fit(tr.select(feats).to_numpy(), tr["y_pc"].to_numpy())
        row = {f: float(tr.select(pl.col(f).last()).item()) for f in feats}
        y0 = float(m.predict([list(row.values())])[0])
        if y0 == 0:
            continue
        rp, rm = row.copy(), row.copy()
        rp[fe] *= 1.1
        rm[fe] *= 0.9
        yp = float(m.predict([list(rp.values())])[0])
        ym = float(m.predict([list(rm.values())])[0])
        acc.append(((yp - ym) / y0) / 0.2)
    return float(np.mean(acc))


print(
    "elasticidades pc (train<=2024, fila base 2024, simetrico +-10%): "
    f"iph={elasticidad_pc('iph_media'):+.3f} "
    f"ocup={elasticidad_pc('ocupacion_media'):+.3f} "
    f"lluvia={elasticidad_pc('lluvia_anual_mm'):+.3f}"
)

elasticidades pc (train<=2024, fila base 2024, simetrico +-10%): iph=+0.275 ocup=+0.023 lluvia=-0.026


## Decisión (2026-09-12)

- **Población como feature: descartada** (empeora el MAPE ~+1,0 pp).
- **Target per cápita: adoptado** con población congelada en el año base (protocolo de producción),
  siempre que la tabla de resultados confirme la mejora ≥ 0,5 pp y el Wilcoxon significativo.
- La palanca de censo del simulador **no cambia**: se mantiene el coeficiente externo 0,81 medido
  por OLS (la normalización per cápita mejora la precisión del baseline, no la semántica del slider).
- Las elasticidades se recalculan en el DAG sobre el modelo per cápita (el hint de IPH del frontend
  se ajusta al valor resultante).
- Limitación conocida: parte de los municipios pequeños/turísticos empeora individualmente aunque
  la media mejora de forma significativa; el desglose por municipio queda en este notebook.